In [ ]:
import numpy as np
import pandas as pd 
import plotly.express as px
from collections import Counter

In [ ]:
#__import__('warnings').filterwarnings("ignore")

In [ ]:
# download the dataset from https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map
FULL_DATA_PATH = "../data/raw/LSWMD.pkl"

df_wafermap = pd.read_pickle(FULL_DATA_PATH)

The following elements are reviewed for dataset analysis:
1. Data summary
2. Distribution analysis
3. Missing value check

Data Summary

In [ ]:
df_wafermap.info()

In [ ]:
df_wafermap.rename(columns={'trianTestLabel': 'trainTestLabel'}, inplace=True)

In [ ]:
df_wafermap.describe()

Distribution analysis

In [ ]:
df_wafermap.head()

In [ ]:
df_wafermap.tail()

From the above we can figure out that there are 47543 lots. A lot is just a collection of wafers. 

But how many wafers are there in a lot?

In [ ]:
TOTAL_LOTS = 47543
# lot_name_1 = "lot17527" # example where there are only 5 wafers in the lot
lot_name_1 = "lot9304" # example where there are 25 wafers in the lot

print(f"For lot : {lot_name_1}")
df_lot_1 = df_wafermap[df_wafermap['lotName'] == lot_name_1]

# the below indicates that the lot has no duplicate waferIndex
print(f"All unique wafers in lot? {df_lot_1['waferIndex'].nunique() == len(df_lot_1)}")

print(f"Number of unique wafers in the lot : {df_lot_1['waferIndex'].nunique()}")
print(df_lot_1["waferIndex"].unique())

The above analysis tells us that generally each lot has 25 wafers. But this is not always the case. Some lots have less than that as well.

Lets look at that in more detail.

In [ ]:
# wafer-count distribution across lots
wafer_counts = df_wafermap.groupby('lotName')['waferIndex'].nunique().reset_index()
wafer_counts.columns = ['lotName', 'uniqueWafers']

print(f"Expected number of lots: {TOTAL_LOTS}. Found: {len(wafer_counts)}")

fig = px.histogram(
    wafer_counts,
    x='uniqueWafers',
    nbins=25,
    title='How many lots have a specific number of unique wafers?',
    color_discrete_sequence=['teal']
)

fig.add_vline(x=25, line_dash='dash', line_color='red', annotation_text='Expected (25)', annotation_position='top left')
fig.update_layout(
    xaxis_title='Number of Unique Wafers',
    yaxis_title='Number of Lots',
    title_x=0.5,
    width=1000,
    height=500
)

In [ ]:
def find_dim(x):
    dim0=x.shape[0]
    dim1=x.shape[1]
    return dim0,dim1
df_wafermap['waferMapDim']=df_wafermap.waferMap.apply(find_dim)
df_wafermap.sample(5)

In [ ]:
uni_waferDim=np.unique(df_wafermap.waferMapDim, return_counts=True)
print(f"Number of different wafer map dimensions: {uni_waferDim[0].shape[0]}")

Below we see the distribution of train/test labels in the dataset

In [ ]:
label_count = df_wafermap['trainTestLabel'].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)
df_labelCount = label_count.reset_index()
df_labelCount.columns = ['datapoint', 'labelCount']

fig = px.histogram(df_labelCount, x='labelCount', nbins=5,
                   title='Distribution of Label Counts per Row',
                   labels={'labelCount': 'Number of Labels'})
fig.update_layout(title_x=0.5)
fig.show()

In [ ]:
total_rows = len(df_labelCount)
zero_label_rows = (df_labelCount['labelCount'] == 0).sum()
percent_missing = (zero_label_rows / total_rows) * 100

print(f"Missing train/test labels: {zero_label_rows}/{total_rows} ({percent_missing:.2f}%)")

From the above we can see that there are no datapoints which can have two labels such as `['training', 'test']`. But there are a huge number (majority even) of datapoints with no labels.

In [ ]:
flat_labels = [label.tolist()[0] for sublist in df_wafermap['trainTestLabel'] for label in sublist]
label_counts = Counter(flat_labels)
fig = px.bar(
    x=list(label_counts.keys()),
    y=list(label_counts.values()),
    title='Distribution of Train/Test Labels in the Dataset',
    labels={'x': 'Label', 'y': 'Count'},
    color_discrete_sequence=['turquoise']
)
fig.update_layout(title_x=0.5)
fig.show()

Below we see the distribution of failure types in the dataset

In [ ]:
failtype_count = df_wafermap['failureType'].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)
df_failtypeCount = failtype_count.reset_index()
df_failtypeCount.columns = ['datapoint', 'failTypeCount']

fig = px.histogram(df_failtypeCount, x='failTypeCount', nbins=5,
                   title='Distribution of Fail Type Counts per Row',
                   labels={'failTypeCount': 'Number of Failure Type Labels'})
fig.update_layout(title_x=0.5)
fig.show()

In [ ]:
total_rows = len(df_failtypeCount)
zero_label_rows = (df_failtypeCount['failTypeCount'] == 0).sum()
percent_missing = (zero_label_rows / total_rows) * 100

print(f"Missing failure type labels: {zero_label_rows}/{total_rows} ({percent_missing:.2f}%)")

As with `trainTestLabels`, we see that each datapoint can only have a single type of failure label associated with it. But a huge number of datapoints exist with no labels.

Interestingly, the number of datapoints with missing train/test label == the number of datapoints with missing failure label

In [ ]:
flat_labels = [label.tolist()[0] for sublist in df_wafermap['failureType'] for label in sublist]
label_counts = Counter(flat_labels)
fig = px.bar(
    x=list(label_counts.keys()),
    y=list(label_counts.values()),
    title='Distribution of Failure Type Labels in the Dataset',
    labels={'x': 'Failure Type Label', 'y': 'Count'},
    color_discrete_sequence=['turquoise']
)
fig.update_layout(title_x=0.5)
fig.show()

In [ ]:
flat_labels = [label.tolist()[0] for sublist in df_wafermap['trainTestLabel'] for label in sublist]
label_counts = Counter(flat_labels)
fig = px.bar(
    x=list(label_counts.keys()),
    y=list(label_counts.values()),
    title='Distribution of Train/Test Labels in the Dataset',
    labels={'x': 'Label', 'y': 'Count'},
    color_discrete_sequence=['purple']
)
fig.show()

Missing value check

Remove elements of the dataset with missing `trainTestLabel` and `failureType`

In [ ]:
print(f"Number of elements in dataset before removing missing values: {df_wafermap.shape[0]}")

df_wafermap['failureNum']=df_wafermap.failureType
df_wafermap['trainTestNum']=df_wafermap.trainTestLabel
mapping_type={'Center':0,'Donut':1,'Edge-Loc':2,'Edge-Ring':3,'Loc':4,'Random':5,'Scratch':6,'Near-full':7,'none':8}
mapping_traintest={'Training':0,'Test':1}
df_wafermap=df_wafermap.replace({'failureNum':mapping_type, 'trainTestNum':mapping_traintest})

print(df_wafermap)

"""df_withlabel = df_wafermap[(df_wafermap['failureNum']>=0) & (df_wafermap['failureNum']<=8)]
df_withlabel =df_withlabel.reset_index()
df_withpattern = df_wafermap[(df_wafermap['failureNum']>=0) & (df_wafermap['failureNum']<=7)]
df_withpattern = df_withpattern.reset_index()
df_nonpattern = df_wafermap[(df_wafermap['failureNum']==8)]
print(f"df_withlabel.shape[0], df_withpattern.shape[0], df_nonpattern.shape[0]")"""